In [ ]:
library(fixest)
library(dplyr)
library(tidyr)
library(ggplot2)
library(lubridate)
library(modelsummary)

panel <- read.csv("../data/monthly_panel.csv")
raw_df <- read.csv("../data/sc_control_data_final_with_agg_score_and_vuln_count.csv")

raw_df <- raw_df %>%
    mutate(year_month = floor_date(as_date(published_at), "month"))


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union



Attaching package: ‘lubridate’


The following objects are masked from ‘package:base’:

    date, intersect, setdiff, union




I would first check whether separate identification is even feasible. Build the co-adoption structure explicitly: the pairwise correlation matrix of the D^k, and check how often each check is adopted alone and whether its adoption timing differs from the others’. If, say, branch protection is almost never turned on without code review turning on in the same release, no estimator can separate them — that’s a data limitation, not a method choice, and the honest move is to report them as a bundle. The checks that do get adopted standalone, at distinct times, are the ones you can actually speak to individually.

## **Without Categorizing the Security Checks**

In [ ]:
metrics <- c(
    "Binary_Artifacts_score", "Code_Review_score",
    "Contrib_score", "Dangerous_Workflow_score", "DependUpTool_score",
    "Fuzzing_score", "License_score",
    "Maintained_score", "Pinned_Dependencies_score",
    "Security_Policy_score", "Token_Permissions_score"
)

controls <- "log1p(downloads_7_day_total) + log1p(dependents) + log1p(loc) + version_age_days + log1p(Release_count)"
fe_part  <- "package_name + year_month"

metrics_joined <- paste(metrics, collapse = " + ")

## Build formula: continuous versions for all the metrics
# Aggregate_score is excluded from this model entirely.
fml_str <- paste("Vulnerability_count ~", metrics_joined, "+", controls, "|", fe_part)
fml     <- as.formula(fml_str)

## Run the model
model_all <- feglm(fml, data = panel, family = "poisson", cluster = ~package_name)

summary(model_all)

NOTES: 26,906 observations removed because of NA values (RHS: 26,906).
       4,402/0 fixed-effects (18,659 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: Vulnerability_count
Observations: 58,489
Fixed-effects: package_name: 7,907,  year_month: 25
Standard-errors: Clustered (package_name) 
                              Estimate Std. Error   z value   Pr(>|z|)    
Binary_Artifacts_score       -0.068930   0.027777 -2.481560 1.3081e-02 *  
Code_Review_score             0.008846   0.004979  1.776780 7.5604e-02 .  
Contrib_score                 0.022952   0.014000  1.639385 1.0113e-01    
Dangerous_Workflow_score     -0.004396   0.005646 -0.778675 4.3617e-01    
DependUpTool_score            0.004923   0.005480  0.898468 3.6894e-01    
Fuzzing_score                 0.024367   0.028686  0.849420 3.9565e-01    
License_score                 0.020302   0.022878  0.887366 3.7488e-01    
Maintained_score             -0.013489   0.002318 -5.819649 5.8971e-09 ***
Pinned_Dependencies_score    -0.023674   0.011582 -2.044010 4.0953e-02 *  
Security_Policy_score         0.004600   0.007962  0.577747 5.6343e-0

## **With Categorized Security Checks**

In [ ]:
# Dichotomize variables 
## worthwile to rerun the tests with x == 0, etc to check sensitivity to the tolerance choice 

bin_zero_pos <- function(x) {              # adoption: 0 vs any positive
  factor(if_else(x < 0.5, "zero", "positive"),
         levels = c("zero", "positive"))
}

bin_ten_less <- function(x) {              # regression: full marks vs not
  factor(if_else(x > 9.5, "ten", "below_ten"),
         levels = c("below_ten", "ten"))
}


binnings <- list(zero_pos = bin_zero_pos, ten_less = bin_ten_less)

chosen_binning <- c(
  # Aggregate_score            = "continuous",
  Maintained_score           = "continuous",
  Code_Review_score          = "continuous",
  Pinned_Dependencies_score  = "zero_pos",
  Security_Policy_score      = "zero_pos",
  Token_Permissions_score    = "zero_pos",
  DependUpTool_score         = "zero_pos",
  Binary_Artifacts_score     = "ten_less",
  Dangerous_Workflow_score   = "ten_less",
  Contrib_score              = "ten_less"
)

binned_metrics <- chosen_binning[chosen_binning != "continuous"]
continuous_metrics <- names(chosen_binning[chosen_binning == "continuous"])

# 1. Create binned columns in panel
panel_binned <- panel
for (m in names(binned_metrics)) {
  f <- binnings[[binned_metrics[[m]]]]
  panel_binned[[paste0(m, "_bin")]] <- f(panel_binned[[m]])
}

# 2. Build formula: binned versions for the binned metrics,
# raw continuous columns for Maintained_score and Code_Review_score
# Aggregate_score is excluded from this model 
bin_terms  <- paste(paste0(names(binned_metrics), "_bin"), collapse = " + ")
cont_terms <- paste(continuous_metrics, collapse = " + ")

fml_all_binned_str <- paste(
  "Vulnerability_count ~",
  paste(c(cont_terms, bin_terms), collapse = " + "), "+",
  controls,
  "|", fe_part
)
fml_all_binned <- as.formula(fml_all_binned_str)

# 3. Run the model
model_all_binned <- feglm(fml_all_binned, data = panel_binned,
                           family = "poisson", cluster = ~package_name)
summary(model_all_binned)

NOTES: 26,860 observations removed because of NA values (RHS: 26,860).
       4,400/0 fixed-effects (18,668 observations) removed because of only 0 outcomes or singletons.



GLM estimation, family = poisson, Dep. Var.: Vulnerability_count
Observations: 58,526
Fixed-effects: package_name: 7,912,  year_month: 25
Standard-errors: Clustered (package_name) 
                                       Estimate Std. Error   z value   Pr(>|z|)
Maintained_score                      -0.013056   0.002728 -4.785682 1.7041e-06
Code_Review_score                      0.009188   0.004845  1.896470 5.7898e-02
Pinned_Dependencies_score_binpositive -0.076173   0.082431 -0.924081 3.5544e-01
Security_Policy_score_binpositive      0.036096   0.070137  0.514653 6.0680e-01
Token_Permissions_score_binpositive    0.044684   0.149444  0.299000 7.6494e-01
DependUpTool_score_binpositive         0.045929   0.057158  0.803534 4.2167e-01
Binary_Artifacts_score_binten         -0.299306   0.088351 -3.387682 7.0486e-04
Dangerous_Workflow_score_binten       -0.044260   0.053612 -0.825557 4.0906e-01
Contrib_score_binten                   0.143553   0.081693  1.757227 7.8879e-02
log1p(downloads_7_d